# Notebook 01b — Hand X-Ray: Balanced Undersampling & Safe Augmentation

**Goal:** Membuat dataset seimbang dari gambar X-ray tangan dengan total ~2000–3000 gambar.

## Strategi
1. **Undersampling:** Sesuaikan kelas mayoritas (Non-fractured) ke jumlah kelas minoritas (Fractured).
2. **Augmentasi Aman Secara Anatomis:** Buat salinan augmentasi untuk setiap gambar agar total mencapai target.
3. **Stratified Split:** Bagi data menjadi Train (70%) / Val (15%) / Test (15%) dengan menjaga rasio kelas.

## Augmentation Strategy (Anatomically Safe)
✅ **SAFE - Diterapkan:**
- **Horizontal Flip** (p=0.5) — Aman untuk X-ray tangan (simetri kiri ↔ kanan)
- **Small Rotation** (±10°) — Mensimulasikan variasi posisi pasien
- **Brightness/Contrast** — Mensimulasikan kualitas mesin X-ray berbeda
- **Mild Translation** (5%) — Perbedaan posisi tangan
- **Mild Scale** (±15%) — Perbedaan ukuran tangan / jarak

❌ **UNSAFE - Dinonaktifkan:**
- **Vertical Flip** — Membuat anatomi tidak benar (tulang terbalik)
- **Shear** — Mendistorsi struktur tulang
- **Perspective** — X-ray adalah proyeksi, bukan perspektif
- **Hue/Saturation** — X-ray bersifat grayscale

## 0. Setup & Konfigurasi

In [ ]:
import os
import cv2
import random
import shutil
import math
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import albumentations as A
import yaml

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR   = Path.cwd().parent
IMAGES_DIR = BASE_DIR / "images"
ANNOT_DIR  = BASE_DIR / "Annotations" / "YOLO"
DATASET_CSV = BASE_DIR / "dataset.csv"
OUTPUT_DIR  = BASE_DIR / "yolo_dataset_hand_undersampled"

# ── Target Configuration ─────────────────────────────────────────────────────
TARGET_TOTAL   = 2628   # Target total gambar setelah augmentasi (~2000–3000)
VAL_RATIO      = 0.15
TEST_RATIO     = 0.15
TRAIN_RATIO    = 1.0 - VAL_RATIO - TEST_RATIO  # 0.70

# ── Create Output Structure ──────────────────────────────────────────────────
for split in ["train", "val", "test"]:
    (OUTPUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

print(f"Base directory  : {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Target total    : {TARGET_TOTAL} images")
print(f"Split ratio     : Train {TRAIN_RATIO:.0%} | Val {VAL_RATIO:.0%} | Test {TEST_RATIO:.0%}")

Base directory  : d:\Project Medical Object Detection\FracAtlas
Output directory: d:\Project Medical Object Detection\FracAtlas\yolo_dataset_balanced
Target total    : 2628 images
Split ratio     : Train 70% | Val 15% | Test 15%


## 1. Load & Analisis Dataset

In [9]:
# Load dan filter hanya gambar tangan
df = pd.read_csv(DATASET_CSV)
hand_df = df[df['hand'] == 1].copy()

n_frac     = len(hand_df[hand_df['fractured'] == 1])
n_non_frac = len(hand_df[hand_df['fractured'] == 0])
n_total    = len(hand_df)

print("=" * 50)
print("DISTRIBUSI DATA HAND (SEBELUM BALANCING)")
print("=" * 50)
print(f"  Fractured        : {n_frac:>6}  ({n_frac/n_total*100:.1f}%)")
print(f"  Non-fractured    : {n_non_frac:>6}  ({n_non_frac/n_total*100:.1f}%)")
print(f"  Total Hand Images: {n_total:>6}")
print("=" * 50)

# Tampilkan kolom untuk validasi
print(f"\nKolom tersedia: {list(df.columns)}")
print(f"Sample data:")
print(hand_df.head(3))

DISTRIBUSI DATA HAND (SEBELUM BALANCING)
  Fractured        :    438  (28.5%)
  Non-fractured    :   1100  (71.5%)
  Total Hand Images:   1538

Kolom tersedia: ['image_id', 'hand', 'leg', 'hip', 'shoulder', 'mixed', 'hardware', 'multiscan', 'fractured', 'fracture_count', 'frontal', 'lateral', 'oblique']
Sample data:
          image_id  hand  leg  hip  shoulder  mixed  hardware  multiscan  \
6   IMG0000006.jpg     1    0    0         0      0         0          1   
7   IMG0000007.jpg     1    0    0         0      0         0          0   
11  IMG0000011.jpg     1    0    0         1      1         0          0   

    fractured  fracture_count  frontal  lateral  oblique  
6           0               0        0        1        1  
7           0               0        0        0        1  
11          0               0        1        0        0  


## 2. Dynamic Undersampling

In [10]:
# Tentukan jumlah minority class secara dinamis
n_min = min(n_frac, n_non_frac)

# Undersample mayoritas ke jumlah minoritas
frac_samples     = hand_df[hand_df['fractured'] == 1].sample(n=n_min, random_state=SEED)
non_frac_samples = hand_df[hand_df['fractured'] == 0].sample(n=n_min, random_state=SEED)

balanced_df  = pd.concat([frac_samples, non_frac_samples]).reset_index(drop=True)
total_base   = len(balanced_df)  # n_min * 2

# Hitung jumlah augmentasi yang diperlukan
# total_output = total_base * (1_original + n_aug)
n_copies_per_image = math.ceil(TARGET_TOTAL / total_base)   # termasuk original
n_aug_per_image    = n_copies_per_image - 1                 # hanya augmentasi
estimated_total    = total_base * n_copies_per_image

print("=" * 50)
print("HASIL UNDERSAMPLING")
print("=" * 50)
print(f"  Minority count         : {n_min}")
print(f"  Fractured  (sampled)   : {len(frac_samples)}")
print(f"  Non-frac   (sampled)   : {len(non_frac_samples)}")
print(f"  Total base images      : {total_base}")
print()
print("=" * 50)
print("RENCANA AUGMENTASI")
print("=" * 50)
print(f"  Target total           : {TARGET_TOTAL}")
print(f"  Copies per image       : 1 original + {n_aug_per_image} augmented")
print(f"  Estimated total output : {estimated_total}")
print(f"    → Fractured  total   : ~{n_min * n_copies_per_image}")
print(f"    → Non-frac   total   : ~{n_min * n_copies_per_image}")

HASIL UNDERSAMPLING
  Minority count         : 438
  Fractured  (sampled)   : 438
  Non-frac   (sampled)   : 438
  Total base images      : 876

RENCANA AUGMENTASI
  Target total           : 2628
  Copies per image       : 1 original + 2 augmented
  Estimated total output : 2628
    → Fractured  total   : ~1314
    → Non-frac   total   : ~1314


## 3. Define Anatomically Safe Augmentation Pipeline

In [11]:
# Pipeline augmentasi yang aman secara anatomis untuk X-ray tangan
aug_pipeline = A.Compose(
    [
        # ✅ SAFE: Horizontal flip — tangan kiri↔kanan masuk akal secara klinis
        A.HorizontalFlip(p=0.5),

        # ✅ SAFE: Rotasi kecil ±10° — variasi posisi pasien
        # ❌ TIDAK menggunakan rotasi 90° / 180° — tidak realistis secara klinis
        A.Affine(
            rotate=(-10, 10),           # rotasi ±10 derajat
            translate_percent=(-0.05, 0.05),  # translasi ±5%
            scale=(0.85, 1.15),         # skala ±15%
            shear=0,                    # ❌ shear = 0 (distorsi tulang)
            mode=cv2.BORDER_CONSTANT,
            cval=0,
            p=0.8
        ),

        # ✅ SAFE: Brightness/Contrast — variasi mesin X-ray berbeda
        A.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2,
            p=0.6
        ),

        # ✅ SAFE: Slight Gaussian blur — variasi ketajaman gambar X-ray
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),

        # ✅ SAFE: Gaussian noise kecil — noise sensor detector
        A.GaussNoise(std_range=(0.005, 0.02), p=0.2),

        # ❌ TIDAK: Vertical Flip (anatomi terbalik)
        # ❌ TIDAK: Shear (distorsi tulang)
        # ❌ TIDAK: Perspective (X-ray bukan foto perspektif)
        # ❌ TIDAK: Hue/Saturation (X-ray grayscale)
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels'],
        min_visibility=0.3     # bbox minimal 30% visible setelah transformasi
    )
)

print("Augmentation pipeline siap.")
print("Transforms aktif:")
print("  ✅ HorizontalFlip  (p=0.5)")
print("  ✅ Affine: rotate±10°, translate±5%, scale±15%, shear=0  (p=0.8)")
print("  ✅ RandomBrightnessContrast ±0.2  (p=0.6)")
print("  ✅ GaussianBlur blur_limit=(3,5)  (p=0.2)")
print("  ✅ GaussNoise std=(0.005,0.02)    (p=0.2)")
print("  ❌ VerticalFlip   — DISABLED")
print("  ❌ Shear          — DISABLED")
print("  ❌ Perspective    — DISABLED")
print("  ❌ Hue/Saturation — DISABLED")

Augmentation pipeline siap.
Transforms aktif:
  ✅ HorizontalFlip  (p=0.5)
  ✅ Affine: rotate±10°, translate±5%, scale±15%, shear=0  (p=0.8)
  ✅ RandomBrightnessContrast ±0.2  (p=0.6)
  ✅ GaussianBlur blur_limit=(3,5)  (p=0.2)
  ✅ GaussNoise std=(0.005,0.02)    (p=0.2)
  ❌ VerticalFlip   — DISABLED
  ❌ Shear          — DISABLED
  ❌ Perspective    — DISABLED
  ❌ Hue/Saturation — DISABLED


C:\Users\alema\AppData\Local\Temp\ipykernel_30648\2234757985.py:9: UserWarning: Argument(s) 'mode, cval' are not valid for transform Affine
  A.Affine(


## 4. Helper Functions

In [12]:
def get_image_and_label_paths(img_id: str):
    """Temukan path gambar (Fractured atau Non_fractured folder) dan label-nya."""
    # image_id di CSV sudah termasuk '.jpg', jadi jangan tambah lagi
    p_frac     = IMAGES_DIR / "Fractured"     / img_id
    p_non_frac = IMAGES_DIR / "Non_fractured" / img_id
    
    img_path   = p_frac if p_frac.exists() else p_non_frac
    
    # Untuk label, kita butuh stem (tanpa .jpg) + .txt
    base_name  = Path(img_id).stem
    label_path = ANNOT_DIR / f"{base_name}.txt"
    
    return img_path, label_path

## 5. Generate Augmented Dataset

In [13]:
# Pastikan folder train bersih sebelum generate
for f in (OUTPUT_DIR / "train" / "images").glob("*.jpg"):
    f.unlink()
for f in (OUTPUT_DIR / "train" / "labels").glob("*.txt"):
    f.unlink()

skipped     = 0
aug_failed  = 0
total_saved = 0

for _, row in tqdm(balanced_df.iterrows(), total=len(balanced_df), desc="Generating images"):
    img_id = row['image_id']
    img_path, label_path = get_image_and_label_paths(img_id)

    # Skip jika gambar tidak ditemukan
    if not img_path.exists():
        skipped += 1
        continue

    # Baca gambar
    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        skipped += 1
        continue
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # Baca label
    bboxes, class_labels = read_yolo_labels(label_path)

    # ── Simpan Original ───────────────────────────────────────────────────────
    out_img = OUTPUT_DIR / "train" / "images" / f"{img_id}_orig.jpg"
    out_lbl = OUTPUT_DIR / "train" / "labels" / f"{img_id}_orig.txt"
    save_image_and_label(image_rgb, bboxes, class_labels, out_img, out_lbl)
    total_saved += 1

    # ── Simpan Augmented Copies ───────────────────────────────────────────────
    for i in range(n_aug_per_image):
        try:
            augmented     = aug_pipeline(image=image_rgb,
                                         bboxes=bboxes,
                                         class_labels=class_labels)
            aug_img_rgb   = augmented['image']
            aug_bboxes    = augmented['bboxes']
            aug_classes   = augmented['class_labels']

            out_img = OUTPUT_DIR / "train" / "images" / f"{img_id}_aug{i}.jpg"
            out_lbl = OUTPUT_DIR / "train" / "labels" / f"{img_id}_aug{i}.txt"
            save_image_and_label(aug_img_rgb, aug_bboxes, aug_classes, out_img, out_lbl)
            total_saved += 1
        except Exception as e:
            aug_failed += 1

print()
print("=" * 50)
print("HASIL GENERASI DATA")
print("=" * 50)
print(f"  Gambar tersimpan  : {total_saved}")
print(f"  Gambar di-skip    : {skipped}  (file tidak ditemukan)")
print(f"  Augmentasi gagal  : {aug_failed}  (bbox hilang / error lain)")

Generating images: 100%|██████████| 876/876 [00:45<00:00, 19.25it/s]


HASIL GENERASI DATA
  Gambar tersimpan  : 2628
  Gambar di-skip    : 0  (file tidak ditemukan)
  Augmentasi gagal  : 0  (bbox hilang / error lain)


## 6. Stratified Train / Val / Test Split

In [14]:
# Kumpulkan semua gambar di folder train
all_imgs = sorted((OUTPUT_DIR / "train" / "images").glob("*.jpg"))

# Pisahkan berdasarkan kelas menggunakan nama file
# Gambar dari balanced_df sudah seimbang; kita split per kelas agar rasio terjaga
frac_imgs     = [p for p in all_imgs if any(
    p.stem.startswith(iid)
    for iid in frac_samples['image_id'].values
)]
non_frac_imgs = [p for p in all_imgs if any(
    p.stem.startswith(iid)
    for iid in non_frac_samples['image_id'].values
)]

random.shuffle(frac_imgs)
random.shuffle(non_frac_imgs)

def split_list(lst, val_r, test_r):
    n_val  = int(len(lst) * val_r)
    n_test = int(len(lst) * test_r)
    return lst[n_val + n_test:], lst[:n_val], lst[n_val:n_val + n_test]

frac_train,     frac_val,     frac_test     = split_list(frac_imgs,     VAL_RATIO, TEST_RATIO)
non_frac_train, non_frac_val, non_frac_test = split_list(non_frac_imgs, VAL_RATIO, TEST_RATIO)

val_imgs  = frac_val  + non_frac_val
test_imgs = frac_test + non_frac_test

def move_split(img_list, target_split):
    """Pindahkan gambar beserta label ke folder split yang ditentukan."""
    for img_p in img_list:
        lbl_src = OUTPUT_DIR / "train" / "labels" / f"{img_p.stem}.txt"
        lbl_dst = OUTPUT_DIR / target_split / "labels" / f"{img_p.stem}.txt"
        img_dst = OUTPUT_DIR / target_split / "images" / img_p.name
        shutil.move(str(img_p), str(img_dst))
        if lbl_src.exists():
            shutil.move(str(lbl_src), str(lbl_dst))

move_split(val_imgs,  "val")
move_split(test_imgs, "test")

# Hitung final
n_train = len(list((OUTPUT_DIR / "train" / "images").glob("*.jpg")))
n_val   = len(list((OUTPUT_DIR / "val"   / "images").glob("*.jpg")))
n_test  = len(list((OUTPUT_DIR / "test"  / "images").glob("*.jpg")))
n_total_final = n_train + n_val + n_test

print("=" * 50)
print("DISTRIBUSI FINAL DATASET")
print("=" * 50)
print(f"  Train : {n_train:>5} gambar  ({n_train/n_total_final*100:.1f}%)")
print(f"  Val   : {n_val:>5} gambar  ({n_val/n_total_final*100:.1f}%)")
print(f"  Test  : {n_test:>5} gambar  ({n_test/n_total_final*100:.1f}%)")
print(f"  Total : {n_total_final:>5} gambar")
print("=" * 50)

DISTRIBUSI FINAL DATASET
  Train :  1840 gambar  (70.0%)
  Val   :   394 gambar  (15.0%)
  Test  :   394 gambar  (15.0%)
  Total :  2628 gambar


## 7. Generate YAML Config untuk YOLOv8

In [15]:
yaml_config = {
    'path': str(OUTPUT_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    1,
    'names': ['fracture']
}

yaml_path = OUTPUT_DIR / "dataset.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_config, f, default_flow_style=False, allow_unicode=True)

print(f"YAML config disimpan: {yaml_path}")
print()
print("Isi dataset.yaml:")
with open(yaml_path) as f:
    print(f.read())

YAML config disimpan: d:\Project Medical Object Detection\FracAtlas\yolo_dataset_balanced\dataset.yaml

Isi dataset.yaml:
names:
- fracture
nc: 1
path: d:\Project Medical Object Detection\FracAtlas\yolo_dataset_balanced
test: test/images
train: train/images
val: val/images



## 8. Verifikasi Akhir

In [16]:
import warnings

print("=" * 60)
print("VERIFIKASI KONSISTENSI DATASET")
print("=" * 60)

issues = []

for split in ["train", "val", "test"]:
    imgs   = set(p.stem for p in (OUTPUT_DIR / split / "images").glob("*.jpg"))
    labels = set(p.stem for p in (OUTPUT_DIR / split / "labels").glob("*.txt"))

    img_no_lbl = imgs - labels
    lbl_no_img = labels - imgs

    print(f"\n[{split.upper()}]")
    print(f"  Images : {len(imgs)}")
    print(f"  Labels : {len(labels)}")

    if img_no_lbl:
        print(f"  ⚠️  {len(img_no_lbl)} gambar TANPA label")
        issues.append(f"{split}: {len(img_no_lbl)} gambar tanpa label")
    else:
        print(f"  ✅ Semua gambar punya label")

    if lbl_no_img:
        print(f"  ⚠️  {len(lbl_no_img)} label TANPA gambar")
        issues.append(f"{split}: {len(lbl_no_img)} label tanpa gambar")
    else:
        print(f"  ✅ Semua label punya gambar")

print()
if issues:
    print("⚠️  Ada isu yang perlu diperiksa:")
    for issue in issues:
        print(f"   - {issue}")
else:
    print("✅ Dataset konsisten! Siap untuk training YOLOv8.")

print()
print("=" * 60)
print("RINGKASAN AKHIR")
print("=" * 60)
print(f"  Output: {OUTPUT_DIR}")
print(f"  YAML  : {yaml_path}")
print(f"  Train : {n_train} | Val: {n_val} | Test: {n_test} | Total: {n_total_final}")
print()
print("Langkah berikutnya: Jalankan Notebook 02 untuk training YOLOv8.")

VERIFIKASI KONSISTENSI DATASET

[TRAIN]
  Images : 1840
  Labels : 1840
  ✅ Semua gambar punya label
  ✅ Semua label punya gambar

[VAL]
  Images : 394
  Labels : 394
  ✅ Semua gambar punya label
  ✅ Semua label punya gambar

[TEST]
  Images : 394
  Labels : 394
  ✅ Semua gambar punya label
  ✅ Semua label punya gambar

✅ Dataset konsisten! Siap untuk training YOLOv8.

RINGKASAN AKHIR
  Output: d:\Project Medical Object Detection\FracAtlas\yolo_dataset_balanced
  YAML  : d:\Project Medical Object Detection\FracAtlas\yolo_dataset_balanced\dataset.yaml
  Train : 1840 | Val: 394 | Test: 394 | Total: 2628

Langkah berikutnya: Jalankan Notebook 02 untuk training YOLOv8.
